# Init

In [180]:
# Standard library imports
from datetime import datetime
import os
import re
import shutil
import sys
import unicodedata
import warnings

# Imports for this notebook
import random

In [181]:
# Should not be defined by user (security)
PATH_LOCATION = '../'
POINTER = '.rf.'

# Dataset split to work with
edit_opt = ['raw', 'processed']

## Helper functions

### Security

In [182]:
# DECORATOR FOR INPUT SANITIZATION
# Safety Mesaures for verfying potential harmful user inputs

def remove_accents(input_str):
    """Replace accents from a string with equivalent letter, issuing a warning if changes are made."""
    original_str = input_str  # Store the original string for comparison
    nfkd_form = unicodedata.normalize('NFKD', input_str)
    result = "".join(c for c in nfkd_form if not unicodedata.combining(c))

    if result != original_str:  # Check if any changes were made
        warnings.warn(f"Accents were removed from input. ('{original_str}' -> '{result}')", UserWarning)

    return result

def sanitize_filename(input_str):
    """Sanitizes a filename by removing accents and disallowed characters."""
    input_str = remove_accents(input_str)  # Remove accents first
    result = input_str

    # Checks for prohibited characters (then deletes them)
    # allowed_chars = r"^[a-zA-Z0-9_\-\.]+$"
    allowed_chars = r"^[a-zA-Z0-9_\-\./]+$" # Allow '/' in user input
    if input_str and not re.fullmatch(allowed_chars, input_str):
        result = re.sub(allowed_chars, "", input_str)  # Sanitize (remove invalid chars)
        warnings.warn(f"This argument is unsafe. A sanitized version will be used instead. ('{input_str}' -> '{result}')", UserWarning)

    #if input_str == result:
    #    print("Original name: '", input_str)
    #    print("Sanitized name: '", result)
    #    print()

    return result

# Decorator wrapper
def validate_filenames():
    def decorator(func):
        def wrapper(*args, **kwargs):
            # Arguments Sanitization 
            sanitized_args = []
            for arg in args:
                if isinstance(arg, str):
                    original_arg = arg
                    sanitized_arg = sanitize_filename(arg) # Sanitize argument
                    if sanitized_arg != original_arg:
                        warnings.warn(f"Argument '{original_arg}' sanitized to '{sanitized_arg}'.", UserWarning)
                else:
                    sanitized_arg = arg
                    # warnings.warn(f"Argument '{arg}' is not string dtype. Be sure the current function can handle it.", UserWarning)
                sanitized_args.append(sanitized_arg)

            # Keyword Arguments Sanitization 
            sanitized_kwargs = {}
            for key, value in kwargs.items():
                if isinstance(value, str):
                    original_value = value
                    sanitized_value = sanitize_filename(value) # Sanitize value for keyword
                    if sanitized_value != original_value:
                        warnings.warn(f"Keyword argument '{key}'='{original_value}' sanitized to '{sanitized_value}'.", UserWarning)
                else:
                    sanitized_value = value
                    # warnings.warn(f"Argument '{value}' is not string dtype. Be sure the current function can handle it.", UserWarning)
                sanitized_kwargs[key] = sanitized_value

            # Return sanitized arguments
            return func(*sanitized_args, **sanitized_kwargs)
        return wrapper
    return decorator


In [183]:
def timestamp(message=''):
    currenttime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    if message:
        print(message, currenttime, sep=": ")
    else:
        print('Timestamp', currenttime, sep=": ")

### File managment

In [184]:
# VERSIÓN MODIFICADA (dir 'out' con PATH)
@validate_filenames()
def build_filename(dset: str = '', dtype: str = '', dir: str='in', name: str | None = None, prefix: str = '', verbose: bool = False) -> str:
    """ 
    Construye el nombre y ruta necesaria para cargar/guardar cada archivo del dataset.

    Parámetros:
        - dset (str): Nombre del subconjunto de datos sobre el que se trabaja ('train', 'valid' o 'test').
        - dtype (str): Tipo de archivo ('image' o 'label').
        - dir (str): Dataset de entrada o de salida ('in' o 'out').
        - name (str, optional): Nombre de archivo (sin extension). Si es None, devuelve la ruta de la carpeta.
        - prefix (str): Establece un prefijo para el nombre del archivo (ej:'tile00x00.').
        - verbose (bool): Ofrece información verbal al usuario.

    Retorna:
        - str: Por defecto entrega la ruta de la carpeta (del dataset de entrada o salida).
        - Si además se incluye 'dset' o 'dtype', entrega el directorio de la subcarpeta correspondiente.
        - Si se incluye un nombre de archivo, devuelve la ruta con el formato adecuado.
    """
    
    # Function Example:
    #    - in: "/data/raw/3.5m.v3i.yolov8/valid/labels/
    #            209_205_50_JPG.rf.a6fdbfed5ddcebe949b5a721c39c6a1f.txt"
    #    - out: "/data/processed/3.5m.v3i.yolov8/valid/labels/
    #            tile00x00.209_205_50_JPG.rf.a6fdbfed5ddcebe949b5a721c39c6a1f.txt"
    #    - shape: /data/{dir}/{PATH}{dsuffix}/{dset}/{dtype}/ <- Folder path
    #             {prefix}.{filename}.{dtype}        <- Full filename

    # INITIALIZER LOGIC
    # Verifies that every input has the correct type
    assert (isinstance(name, str | None)
            and isinstance(dset, str)
            and isinstance(dtype, str)
            and isinstance(prefix, str)
            and isinstance(dir, str)), "❕All arguments must be strings."
    
    try:
        PATH
    except NameError:
        raise NameError("❗️ PATH constant is not defined.")
    try:
        OUTPUT_SUFFIX
    except NameError:
        raise NameError("❗️ OUTPUT_SUFFIX constant is not defined.")
    try:
        POINTER
    except NameError:
        raise NameError("❗️ POINTER constant is not defined.")
    
    # Normalize inputs
    dset = dset.lower()
    dtype = dtype.lower()
    dir = dir.lower()

    # This help splitting the name in half
    # for inserting the prefix next to photo filename (ej: 209_205_50_JPG)
    #POINTER = '.rf.' # could be modified for differente tag name formats
    pointer_idx = name.find(POINTER) if name != None else ""

    # Default folders and extensions
    dset_map = {
        'train': 'train/',
        'valid': 'valid/',
        'test': 'test/',
        '': ''
    }
    # Handle both plural and singular inputs
    dtype_map = {
        'image': ('images', 'jpg'),
        'images': ('images', 'jpg'),
        'label': ('labels', 'txt'),
        'labels': ('labels', 'txt'),
        '': ('', ''),
    }

    def get_dset_path(dset_val: str) -> str:
        if dset_val not in dset_map:
            raise ValueError("❕Invalid 'dset': must be 'train', 'valid', or 'test'. (Empty by default)")
        return dset_map[dset_val]

    def get_dtype_info(dtype_val: str) -> tuple[str, str]:
        if dtype_val not in dtype_map:
            raise ValueError("❕Invalid 'type': must be 'image' or 'label'. (Empty by default)")
        return dtype_map[dtype_val]
    
    # FILENAME CREATION LOGIC
    # Output dataset logic
    if dir == 'out':
        #root = PATH.replace('raw','processed')
        root = PATH
        if OUTPUT_SUFFIX:
            root = f"{root[:-1]}{OUTPUT_SUFFIX}/"

        base_path = f"{root}{get_dset_path(dset)}" if dset else root
        
        if dtype:
            type_folder, extension = get_dtype_info(dtype)
            base_path = f"{base_path}{type_folder}/"
        else:
            type_folder, extension = "",""

        print("✔️ Dataset root created successfully") if verbose else ""
        #return f"{base_path}{filename}" # Ending for 'out' option
    
    # Input dataset logic
    elif dir == 'in':
        root = PATH
        type_folder, extension = get_dtype_info(dtype)
        base_path = f"{root}{get_dset_path(dset)}{type_folder}"
        
    else:
        raise ValueError("❕Invalid 'dir': must be 'in' (default) or 'out'.")
    
    # PREFIX LOGIC
    # Managing filename prefixes
    if name is None or name == '':
        filename = ''
        print("🚨 WARNING: Prefix will be ignored unless a filename is provided.") if prefix else ''
    else:
        if not prefix:
            # If no prefix is defined, it keeps the original name
            filename = f"{name}.{extension}"
        else:
            if pointer_idx == -1:
                # When the reference in name format is not found, it adds the prefix at the end
                filename = f"{name}.{prefix}.{extension}"
                print(f"⚠ The original file does not follow the expected format."
                      f"Filenames should inlcude a '{POINTER}' reference inside.\n"
                      f"Therefore, the indicated prefix value '{prefix}' will be"
                      f"added at the end of the current filename.") if verbose else ''
            else:
                # If the format is ok, it inserts the prefix in between the original JPG filename and the hash part
                name_jpg = name[:pointer_idx]
                name_hash = name[pointer_idx:]
                filename = f"{name_jpg}.{prefix}{name_hash}.{extension}"
    
    #print("Input:", PATH, "Output:", root)
    print("✔️ Filename created succesfully") if verbose else ""
    return f"{base_path}{filename}"

# Core functions

In [185]:
def copy_yaml(input_path, output_dir):
    """
    Copia los archivos de configuración de input_path a output_dir,
    eliminando el contenido existente de output_dir si ya existe.

    Args:
        input_path (str): Ruta de la carpeta de origen.
        output_dir (str): Ruta de la carpeta de destino.
    """

    if not os.path.exists(input_path):
        raise FileNotFoundError(f"⛔️ La carpeta de origen '{input_path}' no existe.")

    try:
        # Elimina el contenido existente de output_dir si existe
        if os.path.exists(output_dir):
            confirmation = input(f"""La carpeta de salida ya existe, para continuar se procederá a eliminar su contenido.
                                 ❓¿Estás seguro de que deseas eliminar el contenido de '{output_dir}'? (y/n):""").lower()
            if confirmation in ['y', 'yes']:
                for file in os.listdir(output_dir):  # Itera sobre los archivos y directorios
                    file_path = os.path.join(output_dir, file)  # Crea la ruta completa

                    if os.path.isfile(file_path):  # Si es un archivo
                        os.remove(file_path)  # Elimina el archivo

                    elif os.path.isdir(file_path):  # Si es un directorio
                        shutil.rmtree(file_path)  # Elimina el directorio y su contenido de forma recursiva
                
                print()
                print("✔️ Contenido de la carpeta eliminado:\n", output_dir)
                print()
            else:
                print(f"❌ Operación cancelada por el usuario. El directorio no se eliminó.\n⏸️ La ejecución se ha detenido.")
                sys.exit()

        # Crea el directorio de salida (ahora vacío)
        os.makedirs(output_dir, exist_ok=True)

        # Itera sobre los elementos en la carpeta raíz de PATH
        for item in os.listdir(input_path):
            item_path = os.path.join(input_path, item)
            dest_path = os.path.join(output_dir, item)

            # Verifica si el elemento es un archivo (no un directorio)
            if os.path.isfile(item_path):
                shutil.copy2(item_path, dest_path)  # Copia el archivo

        print(f"✅ Los archivos de configuración han sido copiados con éxito")

    except OSError as e:
        raise OSError(f" Error inesperado al copiar la carpeta:\n {e}")
    except Exception as e:
        raise Exception(f"❗️ Error inesperado:\n {e}")

In [186]:
def create_list_file(images_path, output_file):
    """
    Crea un archivo de lista con las rutas de las imágenes en un directorio.

    Args:
        images_path (str): Ruta al directorio que contiene las imágenes.
        output_file (str): Ruta al archivo de lista de salida.
    """
    with open(output_file, "w") as f:
        for image_file in os.listdir(images_path):
            f.write(os.path.join(images_path, image_file) + "\n")

In [187]:
def split_dataset(dataset_path: str,
                  output_path: str,
                  train_ratio: float=0.8,
                  val_ratio: float=0.2,
                  create_list: bool=False,
                  seed: int=42
                  ):
    """
    Divide un conjunto de datos de imágenes y etiquetas en subconjuntos de entrenamiento, validación y prueba,
    adecuados para el entrenamiento de modelos de detección de objetos YOLO.
    Los datos deben estar contenidos en la carpeta 'train', organizados en carpetas 'images' y 'labels'.
    Cada archivo de imágen y etiquetas debe tener el mismo nombre.

    Args:
        images_path (str): Ruta al directorio que contiene las imágenes.
        labels_path (str): Ruta al directorio que contiene las etiquetas.
        output_path (str): Ruta al directorio donde se guardarán los conjuntos divididos.
        train_ratio (float): Proporción del conjunto de entrenamiento. (0.8 por defecto)
        val_ratio (float): Proporción del conjunto de validación. (0.2 por defecto)
        create_list (bool): Opción para generar archivos de lista (train.txt, val.txt, test.txt). ('False' por defecto)
        seed (int): Semilla para la generación de números aleatorios. ('42' por defecto)
    """
    # Redondeo de valores
    train_ratio = round(train_ratio, 4)
    val_ratio = round(val_ratio, 4)
    test_ratio = round(1 - (train_ratio + val_ratio), 4)
    #print((train_ratio+val_ratio+test_ratio))

    # Muestra parámetros al usuario
    print('Splitting dataset:')
    print(f' - Train set: {train_ratio}')
    print(f' - Valid set: {val_ratio}')
    print(f' - Test set: {test_ratio}')

    # Establece la semilla.
    random.seed(seed)
    print('Seed:', seed)
    print()
    print(f"'{dataset_path}'\n --> '{output_path}'")
    
    # Construye la ruta de acceso al dataset
    copy_yaml(dataset_path,output_path)
    images_path = os.path.join(dataset_path, "train", "images")
    labels_path = os.path.join(dataset_path, "train", "labels")

    # Crea las rutas de salida para cada split
    train_images_path = os.path.join(output_path, "train", "images")
    val_images_path = os.path.join(output_path, "valid",  "images")
    test_images_path = os.path.join(output_path, "test", "images")
    train_labels_path = os.path.join(output_path, "train", "labels")
    val_labels_path = os.path.join(output_path, "valid", "labels")
    test_labels_path = os.path.join(output_path, "test", "labels")

    # Crea las carpetas si no existen
    os.makedirs(train_images_path, exist_ok=True)
    os.makedirs(val_images_path, exist_ok=True)
    os.makedirs(test_images_path, exist_ok=True) if test_ratio else ''
    os.makedirs(train_labels_path, exist_ok=True)
    os.makedirs(val_labels_path, exist_ok=True)
    os.makedirs(test_labels_path, exist_ok=True) if test_ratio else ''

    # Obtiene la lista de nombres de archivos de imágenes
    image_files = os.listdir(images_path)
    random.shuffle(image_files)  # Mezcla los archivos aleatoriamente.

    # Calcula los índices de división.
    num_images = len(image_files)
    train_split = int(num_images * train_ratio)
    val_split = int(num_images * (train_ratio + val_ratio))
    #test_split = int(num_images * (train_ratio + val_ratio + test_ratio))

    # Divide los archivos en conjuntos de entrenamiento, validación y prueba
    for i, image_file in enumerate(image_files):
        image_path = os.path.join(images_path, image_file)
        label_file = os.path.splitext(image_file)[0] + ".txt"
        label_path = os.path.join(labels_path, label_file)

        # Ignore .DS_Store files
        if image_file == ".DS_Store":
            continue

        if i < train_split:
            shutil.copyfile(image_path, os.path.join(train_images_path, image_file))
            shutil.copyfile(label_path, os.path.join(train_labels_path, label_file))
        elif i < val_split:
            shutil.copyfile(image_path, os.path.join(val_images_path, image_file))
            shutil.copyfile(label_path, os.path.join(val_labels_path, label_file))
        else:
            shutil.copyfile(image_path, os.path.join(test_images_path, image_file))
            shutil.copyfile(label_path, os.path.join(test_labels_path, label_file))
    
    print('✅ División del dataset finalizada con éxito.')

    # Crea los archivos de lista (train.txt, val.txt, test.txt).
    if create_list:
        create_list_file(train_images_path, os.path.join(output_path, "train.txt"))
        create_list_file(val_images_path, os.path.join(output_path, "val.txt"))
        create_list_file(test_images_path, os.path.join(output_path, "test.txt")) if test_ratio else ''
        print('✅ Archivos de lista creados con éxito.')
    
    timestamp()


# Settings
Set every parameter for the dataset you need to process

In [188]:
# CONFIGURAR DATASET A PROCESAR
DATASET_NAME = '5m.v2i.yolov8'
OUTPUT_SUFFIX = '.split' # Optional (leave empty if not needed)
# Se realiza el split de DATASET_NAME creando una copia en el mismo directorio raíz
# y agregando el OUTPUT_SUFFIX al nombre de la carpeta de salida

# Opcional: Se puede realizar el split sobre un dataset ya divido en mosaicos
EDIT = edit_opt[0] # 0: raw / 1: processed'

# Se genera el PATH a procesar
PATH = f'{PATH_LOCATION}data/{EDIT}/{DATASET_NAME}/'

print("Directorio:", PATH)
print("Dataset:", DATASET_NAME)
timestamp('Ssettings updated at')

Directorio: ../data/raw/5m.v2i.yolov8/
Dataset: 5m.v2i.yolov8
Ssettings updated at: 2025-03-13 23:33:36


In [189]:
# El script genera automáticamente los directorios de entrada y de salida para DATASET_NAME
# siguiendo la estructura de carpetas del proyecto ('data/raw/' o ''data/processed/')
dataset_path = build_filename('','',dir='in')
output_path = build_filename('','',dir='out')
# De ser necesario, pueden definirse otras carpetas diferentes como origen o destino

# Si la suma no da 1, se creará automáticamente 'test' split
val_ratio = 0.2 # Definir el ratio de split (ej: 80 train / 20 val)
train_ratio = 1 - val_ratio # Definir un valor distinto a 1 para crear 'test' set

split_dataset(dataset_path, output_path,train_ratio=train_ratio,val_ratio=val_ratio)
# Config opcional:
#   create_list=True, para generar archivos de lista (train.txt, val.txt, test.txt)
#   seed=int, para utilizar otra semilla (42 por defecto)

Splitting dataset:
 - Train set: 0.8
 - Valid set: 0.2
 - Test set: 0.0
Seed: 42

'../data/raw/5m.v2i.yolov8/'
 --> '../data/raw/5m.v2i.yolov8.split/'

✔️ Contenido de la carpeta eliminado:
 ../data/raw/5m.v2i.yolov8.split/

✅ Los archivos de configuración han sido copiados con éxito
✅ División del dataset finalizada con éxito.
Timestamp: 2025-03-13 23:33:38
